# 04 · Agregación y clasificación final

<a href="https://colab.research.google.com/github/manuelarguelles/tyv-demo-colab/blob/main/notebooks/04_agregacion_clasificacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

Último paso: las **siete notas** (una por criterio, del notebook anterior) se
combinan en un **subtotal curricular sobre 60 puntos**, y ese subtotal se
traduce en una **categoría** para el candidato.

**La fórmula (peso por dimensión, no por criterio individual):**

| Dimensión    | Criterios | Peso   |
|--------------|-----------|--------|
| Formación    | 3         | 20 %   |
| Experiencia  | 2         | 25 %   |
| Técnico      | 2         | 15 %   |

$$\text{Subtotal} = \underbrace{\frac{\sum F}{3\times3}\times20}_{Formación} + \underbrace{\frac{\sum E}{2\times3}\times25}_{Experiencia} + \underbrace{\frac{\sum T}{2\times3}\times15}_{Técnico}$$

(cada dimensión se normaliza contra su propio máximo — 3 puntos por
criterio — antes de aplicar el peso).

**Los umbrales de clasificación** (sobre el subtotal expresado en
porcentaje, 0–100 %):

| Rango          | Categoría          |
|----------------|---------------------|
| < 60 %         | No apto             |
| 60 % – 80 %    | Reserva             |
| ≥ 80 %         | Apto para entrevista |

Esta clasificación es **solo del filtro curricular** — la entrevista
personal (hasta 40 puntos adicionales) queda fuera de este proceso.


## 1. Los resultados del notebook anterior (7 niveles, uno por criterio)

In [ ]:
# Reproducimos aquí el resultado de 03_siete_consultas_llm.ipynb (modo simulado)
# para que este notebook sea ejecutable de forma independiente.
RUBRICA_DIMENSIONES = {
    "F_01": "Formación", "F_02": "Formación", "F_03": "Formación",
    "E_01": "Experiencia", "E_02": "Experiencia",
    "T_01": "Técnico", "T_02": "Técnico",
}

resultados = {
    "F_01": {"valor": 2, "cita_verificada": True},
    "F_02": {"valor": 3, "cita_verificada": True},
    "F_03": {"valor": 2, "cita_verificada": True},
    "E_01": {"valor": 3, "cita_verificada": True},
    "E_02": {"valor": 3, "cita_verificada": True},
    "T_01": {"valor": 2, "cita_verificada": True},
    "T_02": {"valor": 2, "cita_verificada": True},
}
for cid, r in resultados.items():
    print(f"{cid}  ({RUBRICA_DIMENSIONES[cid]:12s}) → nivel {r['valor']}")


## 2. Agregación: subtotal sobre 60

In [ ]:
from decimal import Decimal, ROUND_HALF_UP

# (columnas del grupo, peso %) — igual que el sistema real
GRUPOS = {
    "Formación":   (["F_01", "F_02", "F_03"], 20),
    "Experiencia": (["E_01", "E_02"],         25),
    "Técnico":     (["T_01", "T_02"],         15),
}

def calcular_subtotal(resultados: dict, grupos: dict = GRUPOS) -> dict:
    """Normaliza cada dimensión contra su propio máximo (3 pts × N criterios)
    y aplica su peso. Si falta algún nivel en una dimensión, esa dimensión
    (y por lo tanto el total) queda en None — el sistema real NUNCA rellena
    un hueco con un promedio o un valor por defecto."""
    dimensiones, faltantes = {}, []
    for dimension, (criterios, peso) in grupos.items():
        valores = []
        for cid in criterios:
            v = resultados.get(cid, {}).get("valor")
            if v not in (1, 2, 3):
                faltantes.append(cid)
            else:
                valores.append(v)
        if len(valores) == len(criterios):
            puntos = (Decimal(sum(valores)) * peso / (3 * len(criterios))).quantize(
                Decimal("0.1"), rounding=ROUND_HALF_UP)
            dimensiones[dimension] = float(puntos)
        else:
            dimensiones[dimension] = None
    total = None if faltantes else round(sum(dimensiones.values()), 1)
    return {"total": total, "maximo": 60, "dimensiones": dimensiones, "faltantes": faltantes}

calculo = calcular_subtotal(resultados)
for dimension, puntos in calculo["dimensiones"].items():
    print(f"{dimension:12s} → {puntos} / {GRUPOS[dimension][1]}")
print(f"\nSubtotal curricular: {calculo['total']} / 60")


## 3. Clasificación final

In [ ]:
CLASES = ("no apto", "reserva", "apto entrevista")

def clasificar(subtotal: float | None, umbral_reserva: float = 60, umbral_apto: float = 80) -> dict:
    if subtotal is None:
        return {"porcentaje": None, "etiqueta": None}
    if not 0 <= subtotal <= 60:
        raise ValueError("El subtotal curricular debe estar entre 0 y 60")
    porcentaje = subtotal * 100 / 60
    if porcentaje < umbral_reserva:
        etiqueta = CLASES[0]
    elif porcentaje < umbral_apto:
        etiqueta = CLASES[1]
    else:
        etiqueta = CLASES[2]
    return {"porcentaje": round(porcentaje, 1), "etiqueta": etiqueta}

clasificacion = clasificar(calculo["total"])
print(f"Puntaje final: {calculo['total']} / 60  →  {clasificacion['porcentaje']}%")
print(f"Categoría: {clasificacion['etiqueta'].upper()}")


## 4. Reporte final (lo que quedaría en el registro auditable)

In [ ]:
informe = {
    "subtotal_curricular": calculo["total"],
    "maximo": 60,
    "porcentaje": clasificacion["porcentaje"],
    "categoria": clasificacion["etiqueta"],
    "dimensiones": calculo["dimensiones"],
    "detalle_por_criterio": {
        cid: {"dimension": RUBRICA_DIMENSIONES[cid], **resultados[cid]}
        for cid in resultados
    },
}
import json
print(json.dumps(informe, indent=2, ensure_ascii=False))


## 5. Sensibilidad: cómo cambia la categoría según el desempeño

Para entender los umbrales, veamos qué categoría resultaría con distintos
perfiles de desempeño (todo 1, todo 2, todo 3, y uno mixto) — no son
candidatos reales, son solo puntos de referencia sobre la escala.

In [ ]:
perfiles = {
    "Todo nivel 1 (mínimo)": {cid: 1 for cid in RUBRICA_DIMENSIONES},
    "Todo nivel 2 (cumple)": {cid: 2 for cid in RUBRICA_DIMENSIONES},
    "Todo nivel 3 (máximo)": {cid: 3 for cid in RUBRICA_DIMENSIONES},
    "Nuestro ejemplo (mixto)": {cid: resultados[cid]["valor"] for cid in RUBRICA_DIMENSIONES},
}

print(f"{'Perfil':28s} {'Subtotal':>10s} {'%':>7s}  Categoría")
for nombre, niveles in perfiles.items():
    r = {cid: {"valor": v} for cid, v in niveles.items()}
    c = calcular_subtotal(r)
    cl = clasificar(c["total"])
    print(f"{nombre:28s} {c['total']:>10} {cl['porcentaje']:>6}%  {cl['etiqueta']}")


## Fin del recorrido paso a paso

Este notebook cierra el pipeline de 4 etapas. Para ver el flujo **completo,
de punta a punta, en un solo notebook**, abrí `00_pipeline_completo.ipynb`.

---
*Este material es contenido educativo de apoyo a una tesis de maestría (Terry & Valdez — sistema de filtrado curricular). El CV usado es 100% ficticio, construido para esta demostración. Ningún dato de candidatos reales del proyecto se publica en este repositorio: ver `materiales/README.md` para trabajar con datos reales de forma local.*
